# Leave Agent Test Notebook

This notebook tests the `smol_leave_agent.py` and `leave_tools.py` implementation, including:
- Direct tool function testing
- Agent interaction testing
- Error handling
- Full workflow testing (draft -> submit)
- Edge cases and validation


In [5]:
# Setup: Import dependencies and load environment
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Add project root to Python path (works in Jupyter notebooks)
# Strategy: Walk up the directory tree until we find the project root (where 'src' directory exists)
current_dir = Path.cwd().resolve()
project_root = current_dir

# Walk up the directory tree to find project root
max_levels = 5  # Prevent infinite loops
for _ in range(max_levels):
    if (project_root / 'src').exists() and (project_root / 'src' / 'agents').exists():
        # Found project root (has src/agents directory)
        break
    parent = project_root.parent
    if parent == project_root:
        # Reached filesystem root
        break
    project_root = parent
else:
    # If we didn't break, use current directory as fallback
    project_root = current_dir

# Add to path if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"✓ Added project root to path: {project_root}")
else:
    print(f"✓ Project root already in path: {project_root}")

# Verify we can find the src directory
if not (project_root / 'src').exists():
    print(f"⚠ WARNING: Could not find 'src' directory in {project_root}")
    print(f"   Current working directory: {current_dir}")

# Load environment variables (try both project root and current dir)
env_file = project_root / '.env'
if env_file.exists():
    load_dotenv(env_file, override=True)
else:
    load_dotenv(override=True)

# Verify API key is set
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"✓ OpenAI API Key loaded (starts with: {api_key[:8]}...)")
else:
    print("✗ WARNING: OPENAI_API_KEY not set in environment")


✓ Project root already in path: C:\projects\simpliAsk\simpliAsk
✓ OpenAI API Key loaded (starts with: sk-proj-...)


In [6]:
# Import the agent and tools
from src.agents.smol_leave_agent import leave_smol_agent
from src.tools import leave_tools as lt

print("✓ Leave agent and tools imported successfully")
print(f"✓ Agent name: {leave_smol_agent.name}")
print(f"✓ Number of tools: {len(leave_smol_agent.tools)}")

# Display tool names (handle different tool representations)
tool_names = []
for tool in leave_smol_agent.tools:
    if hasattr(tool, 'name'):
        tool_names.append(tool.name)
    elif hasattr(tool, '__name__'):
        tool_names.append(tool.__name__)
    elif isinstance(tool, str):
        tool_names.append(tool)
    else:
        tool_names.append(str(tool))
print(f"✓ Tools: {tool_names}")


✓ Leave agent and tools imported successfully
✓ Agent name: leave_specialist
✓ Number of tools: 7
✓ Tools: ['get_leave_balance_tool', 'draft_leave_request_tool', 'submit_draft_leave_request_tool', 'submit_leave_request_tool', 'list_leave_drafts_tool', 'check_leave_status_tool', 'final_answer']


## 1. Direct Tool Testing

Test the underlying tool functions directly before testing the agent.


### Test 1.1: Get Leave Balance


In [7]:
# Test 1.1: Get leave balance for mark_tan
employee_id = "mark_tan"

# Test annual leave balance
result = lt.get_leave_balance(employee_id, "annual")
print("Annual Leave Balance:")
print(json.dumps(json.loads(result), indent=2))

# Test medical leave balance
result = lt.get_leave_balance(employee_id, "medical")
print("\nMedical Leave Balance:")
print(json.dumps(json.loads(result), indent=2))

# Test family leave balance
result = lt.get_leave_balance(employee_id, "family")
print("\nFamily Leave Balance:")
print(json.dumps(json.loads(result), indent=2))


Annual Leave Balance:
{
  "balance": 14,
  "unit": "days"
}

Medical Leave Balance:
{
  "balance": 14,
  "unit": "days"
}

Family Leave Balance:
{
  "balance": 3,
  "unit": "days"
}


### Test 1.2: Create Draft Leave Request


In [8]:
# Test 1.2: Create a draft leave request
draft_result = lt.draft_leave_request(
    employee_id=employee_id,
    leave_type="annual",
    start_date="2025-12-04",
    end_date="2025-12-10"
)

draft_data = json.loads(draft_result)
print("Draft Leave Request Result:")
print(json.dumps(draft_data, indent=2))

# Extract draft_id for later tests
if "draft_id" in draft_data:
    draft_id = draft_data["draft_id"]
    print(f"\n✓ Draft created with draft_id: {draft_id}")
else:
    print("\n✗ Error: No draft_id in response")
    draft_id = None


--- SYSTEM: Draft created for mark_tan (annual leave) from 2025-12-04 to 2025-12-10 (7 days) ---
Draft Leave Request Result:
{
  "status": "draft",
  "message": "Draft leave request created.",
  "draft_id": "draft-61654",
  "draft": {
    "draft_id": "draft-61654",
    "employee_id": "mark_tan",
    "leave_type": "annual",
    "start_date": "2025-12-04",
    "end_date": "2025-12-10",
    "days": 7,
    "status": "draft"
  }
}

✓ Draft created with draft_id: draft-61654


### Test 1.3: List Leave Drafts


In [9]:
# Test 1.3: List all drafts for the employee
result = lt.list_leave_drafts(employee_id)
drafts_data = json.loads(result)
print(f"Total drafts for {employee_id}: {len(drafts_data.get('drafts', []))}")
print("\nDrafts list:")
print(json.dumps(drafts_data, indent=2))


Total drafts for mark_tan: 1

Drafts list:
{
  "employee_id": "mark_tan",
  "drafts": [
    {
      "draft_id": "draft-61654",
      "employee_id": "mark_tan",
      "leave_type": "annual",
      "start_date": "2025-12-04",
      "end_date": "2025-12-10",
      "days": 7,
      "status": "draft"
    }
  ]
}


### Test 1.4: Submit Draft Leave Request


In [10]:
# Test 1.4: Submit the draft leave request
if draft_id:
    submit_result = lt.submit_draft_leave_request(employee_id, draft_id)
    submit_data = json.loads(submit_result)
    print("Submit Draft Result:")
    print(json.dumps(submit_data, indent=2))
    
    # Extract request_id for later tests
    if "request_id" in submit_data:
        request_id = submit_data["request_id"]
        print(f"\n✓ Request submitted with request_id: {request_id}")
    else:
        print("\n✗ Error: No request_id in response")
        request_id = None
else:
    print("✗ Skipping: No draft_id available")
    request_id = None


--- SYSTEM: Submitting draft draft-61654 as MW19740 - 7 days of annual leave for mark_tan (status: pending) ---
Submit Draft Result:
{
  "status": "success",
  "request_id": "MW19740",
  "leave_status": "pending",
  "message": "Request ID #MW19740 submitted and pending manager review."
}

✓ Request submitted with request_id: MW19740


### Test 1.5: Check Leave Status


In [11]:
# Test 1.5: Check the status of the submitted leave request
if request_id:
    status_result = lt.check_leave_status(request_id)
    status_data = json.loads(status_result)
    print("Leave Status Result:")
    print(json.dumps(status_data, indent=2))
else:
    print("✗ Skipping: No request_id available")


Leave Status Result:
{
  "request_id": "MW19740",
  "status": "pending",
  "employee_id": "mark_tan",
  "leave_type": "annual",
  "days": 7
}


### Test 1.6: Direct Submit (Alternative Method)


In [12]:
# Test 1.6: Direct submit without draft (alternative method)
direct_submit_result = lt.submit_leave_request(
    employee_id=employee_id,
    leave_type="medical",
    days=2
)

direct_submit_data = json.loads(direct_submit_result)
print("Direct Submit Result:")
print(json.dumps(direct_submit_data, indent=2))

if "request_id" in direct_submit_data:
    direct_request_id = direct_submit_data["request_id"]
    print(f"\n✓ Direct request submitted with request_id: {direct_request_id}")
else:
    direct_request_id = None


--- SYSTEM: Submitting 2 days of medical leave for mark_tan with request ID MW96981 (status: pending) ---
Direct Submit Result:
{
  "status": "success",
  "request_id": "MW96981",
  "leave_status": "pending",
  "message": "Request ID #MW96981 submitted and pending manager review."
}

✓ Direct request submitted with request_id: MW96981


## 2. Error Handling Tests

Test various error cases and edge conditions.


### Test 2.1: Invalid Leave Type


In [13]:
# Test 2.1: Invalid leave type
error_result = lt.get_leave_balance(employee_id, "invalid_type")
print("Error Test (invalid leave type):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (invalid leave type):
{
  "error": "Invalid leave type: invalid_type. Must be one of: annual, medical, family"
}


### Test 2.2: Invalid Date Format


In [14]:
# Test 2.2: Invalid date format
error_result = lt.draft_leave_request(
    employee_id=employee_id,
    leave_type="annual",
    start_date="2025/12/04",  # Wrong format
    end_date="2025-12-10"
)
print("Error Test (invalid date format):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (invalid date format):
{
  "error": "Invalid start_date format. Use YYYY-MM-DD."
}


### Test 2.3: End Date Before Start Date


In [15]:
# Test 2.3: End date before start date
error_result = lt.draft_leave_request(
    employee_id=employee_id,
    leave_type="annual",
    start_date="2025-12-10",
    end_date="2025-12-04"  # End before start
)
print("Error Test (end date before start date):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (end date before start date):
{
  "error": "End date must be after or equal to start date."
}


### Test 2.4: Insufficient Leave Balance


In [16]:
# Test 2.4: Insufficient leave balance (request more than available)
# First check current balance
balance_result = lt.get_leave_balance(employee_id, "annual")
balance_data = json.loads(balance_result)
available_days = balance_data.get("balance", 0)
print(f"Current annual leave balance: {available_days} days")

# Try to request more than available
error_result = lt.draft_leave_request(
    employee_id=employee_id,
    leave_type="annual",
    start_date="2025-12-01",
    end_date=f"2025-12-{1 + available_days + 5}"  # Request more than available
)
print("\nError Test (insufficient balance):")
print(json.dumps(json.loads(error_result), indent=2))


Current annual leave balance: 14 days

Error Test (insufficient balance):
{
  "error": "Insufficient leave balance. Requested: 20 days, Available: 14 days"
}


### Test 2.5: Non-existent Employee


In [17]:
# Test 2.5: Non-existent employee
error_result = lt.get_leave_balance("non_existent_employee", "annual")
print("Error Test (non-existent employee):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (non-existent employee):
{
  "error": "Employee not found"
}


### Test 2.6: Non-existent Draft ID


In [18]:
# Test 2.6: Submit non-existent draft
error_result = lt.submit_draft_leave_request(employee_id, "draft-99999")
print("Error Test (non-existent draft):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (non-existent draft):
{
  "error": "Draft not found"
}


### Test 2.7: Non-existent Request ID


In [19]:
# Test 2.7: Check status of non-existent request
error_result = lt.check_leave_status("MW99999")
print("Error Test (non-existent request ID):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (non-existent request ID):
{
  "error": "Request not found",
  "request_id": "MW99999"
}


### Test 2.8: Invalid Days (Zero or Negative)


In [20]:
# Test 2.8: Invalid days (zero or negative)
error_result = lt.submit_leave_request(employee_id, "annual", 0)
print("Error Test (zero days):")
print(json.dumps(json.loads(error_result), indent=2))

error_result = lt.submit_leave_request(employee_id, "annual", -5)
print("\nError Test (negative days):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (zero days):
{
  "error": "Days must be a positive integer"
}

Error Test (negative days):
{
  "error": "Days must be a positive integer"
}


## 3. Edge Cases and Special Scenarios


### Test 3.1: Single Day Leave Request


In [21]:
# Test 3.1: Single day leave (start_date == end_date)
single_day_result = lt.draft_leave_request(
    employee_id=employee_id,
    leave_type="medical",
    start_date="2025-11-15",
    end_date="2025-11-15"  # Same day
)

single_day_data = json.loads(single_day_result)
print("Single Day Leave Request:")
print(json.dumps(single_day_data, indent=2))
if "draft" in single_day_data:
    print(f"\nDays calculated: {single_day_data['draft']['days']}")


--- SYSTEM: Draft created for mark_tan (medical leave) from 2025-11-15 to 2025-11-15 (1 days) ---
Single Day Leave Request:
{
  "status": "draft",
  "message": "Draft leave request created.",
  "draft_id": "draft-93507",
  "draft": {
    "draft_id": "draft-93507",
    "employee_id": "mark_tan",
    "leave_type": "medical",
    "start_date": "2025-11-15",
    "end_date": "2025-11-15",
    "days": 1,
    "status": "draft"
  }
}

Days calculated: 1


### Test 3.2: Multiple Drafts for Same Employee


In [22]:
# Test 3.2: Create multiple drafts and verify they're all listed
draft1 = lt.draft_leave_request(employee_id, "family", "2025-11-20", "2025-11-21")
draft2 = lt.draft_leave_request(employee_id, "annual", "2025-11-25", "2025-11-27")

print("Draft 1 (Family):")
print(json.dumps(json.loads(draft1), indent=2))
print("\nDraft 2 (Annual):")
print(json.dumps(json.loads(draft2), indent=2))

# List all drafts
all_drafts = lt.list_leave_drafts(employee_id)
print("\nAll Drafts:")
print(json.dumps(json.loads(all_drafts), indent=2))


--- SYSTEM: Draft created for mark_tan (family leave) from 2025-11-20 to 2025-11-21 (2 days) ---
--- SYSTEM: Draft created for mark_tan (annual leave) from 2025-11-25 to 2025-11-27 (3 days) ---
Draft 1 (Family):
{
  "status": "draft",
  "message": "Draft leave request created.",
  "draft_id": "draft-77277",
  "draft": {
    "draft_id": "draft-77277",
    "employee_id": "mark_tan",
    "leave_type": "family",
    "start_date": "2025-11-20",
    "end_date": "2025-11-21",
    "days": 2,
    "status": "draft"
  }
}

Draft 2 (Annual):
{
  "status": "draft",
  "message": "Draft leave request created.",
  "draft_id": "draft-63028",
  "draft": {
    "draft_id": "draft-63028",
    "employee_id": "mark_tan",
    "leave_type": "annual",
    "start_date": "2025-11-25",
    "end_date": "2025-11-27",
    "days": 3,
    "status": "draft"
  }
}

All Drafts:
{
  "employee_id": "mark_tan",
  "drafts": [
    {
      "draft_id": "draft-61654",
      "employee_id": "mark_tan",
      "leave_type": "annual",

### Test 3.3: Different Employee


In [23]:
# Test 3.3: Test with different employee (jane_doe)
other_employee = "jane_doe"
other_balance = lt.get_leave_balance(other_employee, "annual")
print(f"Balance for {other_employee}:")
print(json.dumps(json.loads(other_balance), indent=2))

other_draft = lt.draft_leave_request(other_employee, "annual", "2025-12-01", "2025-12-02")
print(f"\nDraft for {other_employee}:")
print(json.dumps(json.loads(other_draft), indent=2))


Balance for jane_doe:
{
  "balance": 2,
  "unit": "days"
}
--- SYSTEM: Draft created for jane_doe (annual leave) from 2025-12-01 to 2025-12-02 (2 days) ---

Draft for jane_doe:
{
  "status": "draft",
  "message": "Draft leave request created.",
  "draft_id": "draft-31007",
  "draft": {
    "draft_id": "draft-31007",
    "employee_id": "jane_doe",
    "leave_type": "annual",
    "start_date": "2025-12-01",
    "end_date": "2025-12-02",
    "days": 2,
    "status": "draft"
  }
}


## 4. Agent Interaction Tests

Test the agent's ability to handle natural language queries and use tools appropriately.


### Test 4.1: Simple Balance Query


In [24]:
# Test 4.1: Agent should check leave balance
response = leave_smol_agent.run("How many days of annual leave do I have left?")
print("Agent Response:")
print(response)


╭────────────────────────────────────────── New run - leave_specialist ───────────────────────────────────────────╮
│                                                                                                                 │
│ How many days of annual leave do I have left?                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_leave_balance_tool' with arguments: {'employee_id': 'mark_tan', 'leave_type': 'annual'}      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"balance": 14, "unit": "days"}

[Step 1: Duration 2.18 seconds| Input tokens: 2,752 | Output tokens: 24]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'You have 14 days of annual leave left.'}               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: You have 14 days of annual leave left.

Final answer: You have 14 days of annual leave left.

[Step 2: Duration 1.12 seconds| Input tokens: 5,589 | Output tokens: 47]

Agent Response:
You have 14 days of annual leave left.


### Test 4.2: Draft Creation Request (with dates)


In [25]:
# Test 4.2: Agent should create a draft when dates are provided
response = leave_smol_agent.run(
    "I would like to apply for 3 days of annual leave from 2025-12-15 to 2025-12-17"
)
print("Agent Response:")
print(response)


╭────────────────────────────────────────── New run - leave_specialist ───────────────────────────────────────────╮
│                                                                                                                 │
│ I would like to apply for 3 days of annual leave from 2025-12-15 to 2025-12-17                                  │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'draft_leave_request_tool' with arguments: {'employee_id': 'mark_tan', 'leave_type': 'annual',    │
│ 'start_date': '2025-12-15', 'end_date': '2025-12-17'}                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- SYSTEM: Draft created for mark_tan (annual leave) from 2025-12-15 to 2025-12-17 (3 days) ---


Observations: {"status": "draft", "message": "Draft leave request created.", "draft_id": "draft-29082", "draft": 
{"draft_id": "draft-29082", "employee_id": "mark_tan", "leave_type": "annual", "start_date": "2025-12-15", 
"end_date": "2025-12-17", "days": 3, "status": "draft"}}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'I have created a draft leave request for 3 days of     │
│ annual leave from 2025-12-15 to 2025-12-17. Your draft ID is draft-29082. Would you like me to submit this      │
│ leave request now?'}                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: I have created a draft leave request for 3 days of annual leave from 2025-12-15 to 2025-12-17. Your 
draft ID is draft-29082. Would you like me to submit this leave request now?

Final answer: I have created a draft leave request for 3 days of annual leave from 2025-12-15 to 2025-12-17. Your 
draft ID is draft-29082. Would you like me to submit this leave request now?

[Step 2: Duration 1.77 seconds| Input tokens: 5,729 | Output tokens: 109]

Agent Response:
I have created a draft leave request for 3 days of annual leave from 2025-12-15 to 2025-12-17. Your draft ID is draft-29082. Would you like me to submit this leave request now?


### Test 4.3: Draft Creation Request (without dates - should ask)


In [26]:
# Test 4.3: Agent should ask for dates when not provided
response = leave_smol_agent.run("I would like to apply for 5 days of annual leave")
print("Agent Response:")
print(response)


╭────────────────────────────────────────── New run - leave_specialist ───────────────────────────────────────────╮
│                                                                                                                 │
│ I would like to apply for 5 days of annual leave                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Please provide the start date for your 5 days of       │
│ annual leave (in YYYY-MM-DD format).'}                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Please provide the start date for your 5 days of annual leave (in YYYY-MM-DD format).

Final answer: Please provide the start date for your 5 days of annual leave (in YYYY-MM-DD format).

[Step 1: Duration 0.81 seconds| Input tokens: 2,753 | Output tokens: 33]

Agent Response:
Please provide the start date for your 5 days of annual leave (in YYYY-MM-DD format).


### Test 4.4: List Drafts Request


In [27]:
# Test 4.4: Agent should list drafts
response = leave_smol_agent.run("Show me all my draft leave requests")
print("Agent Response:")
print(response)


╭────────────────────────────────────────── New run - leave_specialist ───────────────────────────────────────────╮
│                                                                                                                 │
│ Show me all my draft leave requests                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'list_leave_drafts_tool' with arguments: {'employee_id': 'mark_tan'}                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"employee_id": "mark_tan", "drafts": |{"draft_id": "draft-61654", "employee_id": "mark_tan", 
"leave_type": "annual", "start_date": "2025-12-04", "end_date": "2025-12-10", "days": 7, "status": "submitted"}, 
{"draft_id": "draft-93507", "employee_id": "mark_tan", "leave_type": "medical", "start_date": "2025-11-15", 
"end_date": "2025-11-15", "days": 1, "status": "draft"}, {"draft_id": "draft-77277", "employee_id": "mark_tan", 
"leave_type": "family", "start_date": "2025-11-20", "end_date": "2025-11-21", "days": 2, "status": "draft"}, 
{"draft_id": "draft-63028", "employee_id": "mark_tan", "leave_type": "annual", "start_date": "2025-11-25", 
"end_date": "2025-11-27", "days": 3, "status": "draft"}, {"draft_id": "draft-29082", "employee_id": "mark_tan", 
"leave_type": "annual", "start_date": "2025-12-15", "end_date": "2025-12-17", "days": 3, "status": "draft"}]}

[Step 1: Duration 1.20 seconds| Input tokens: 2,748 | Output tokens: 20]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'You have the following draft leave requests:\n1. Draft │
│ ID draft-93507: Medical leave on 2025-11-15 for 1 day (Status: draft)\n2. Draft ID draft-77277: Family leave    │
│ from 2025-11-20 to 2025-11-21 for 2 days (Status: draft)\n3. Draft ID draft-63028: Annual leave from 2025-11-25 │
│ to 2025-11-27 for 3 days (Status: draft)\n4. Draft ID draft-29082: Annual leave from 2025-12-15 to 2025-12-17   │
│ for 3 days (Status: draft)\n\nAlso, you have a previously submitted leave draft:\n- Draft ID draft-61654:       │
│ Annual leave from 2025-12-04 to 2025-12-10 for 7 days (Status: submitted)'}                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: You have the following draft leave requests:
1. Draft ID draft-93507: Medical leave on 2025-11-15 for 1 day (Status: draft)
2. Draft ID draft-77277: Family leave from 2025-11-20 to 2025-11-21 for 2 days (Status: draft)
3. Draft ID draft-63028: Annual leave from 2025-11-25 to 2025-11-27 for 3 days (Status: draft)
4. Draft ID draft-29082: Annual leave from 2025-12-15 to 2025-12-17 for 3 days (Status: draft)

Also, you have a previously submitted leave draft:
- Draft ID draft-61654: Annual leave from 2025-12-04 to 2025-12-10 for 7 days (Status: submitted)

Final answer: You have the following draft leave requests:
1. Draft ID draft-93507: Medical leave on 2025-11-15 for 1 day (Status: draft)
2. Draft ID draft-77277: Family leave from 2025-11-20 to 2025-11-21 for 2 days (Status: draft)
3. Draft ID draft-63028: Annual leave from 2025-11-25 to 2025-11-27 for 3 days (Status: draft)
4. Draft ID draft-29082: Annual leave from 2025-12-15 to 2025-12-17 for 3 days (Status: draft)

Also, you have a previously submitted leave draft:
- Draft ID draft-61654: Annual leave from 2025-12-04 to 2025-12-10 for 7 days (Status: submitted)

[Step 2: Duration 2.80 seconds| Input tokens: 5,888 | Output tokens: 229]

Agent Response:
You have the following draft leave requests:
1. Draft ID draft-93507: Medical leave on 2025-11-15 for 1 day (Status: draft)
2. Draft ID draft-77277: Family leave from 2025-11-20 to 2025-11-21 for 2 days (Status: draft)
3. Draft ID draft-63028: Annual leave from 2025-11-25 to 2025-11-27 for 3 days (Status: draft)
4. Draft ID draft-29082: Annual leave from 2025-12-15 to 2025-12-17 for 3 days (Status: draft)

Also, you have a previously submitted leave draft:
- Draft ID draft-61654: Annual leave from 2025-12-04 to 2025-12-10 for 7 days (Status: submitted)


### Test 4.5: Status Check Request


In [28]:
# Test 4.5: Agent should check leave status (if we have a request_id from earlier)
# Note: This will only work if we have a valid request_id from previous tests
if 'request_id' in locals() and request_id:
    response = leave_smol_agent.run(f"What is the status of my leave request {request_id}?")
    print("Agent Response:")
    print(response)
else:
    print("Skipping: No request_id available from previous tests")


╭────────────────────────────────────────── New run - leave_specialist ───────────────────────────────────────────╮
│                                                                                                                 │
│ What is the status of my leave request MW19740?                                                                 │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'check_leave_status_tool' with arguments: {'request_id': 'MW19740'}                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"request_id": "MW19740", "status": "pending", "employee_id": "mark_tan", "leave_type": "annual", 
"days": 7}

[Step 1: Duration 0.66 seconds| Input tokens: 2,753 | Output tokens: 19]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The status of your leave request MW19740 is pending.   │
│ It is for 7 days of annual leave.'}                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The status of your leave request MW19740 is pending. It is for 7 days of annual leave.

Final answer: The status of your leave request MW19740 is pending. It is for 7 days of annual leave.

[Step 2: Duration 0.93 seconds| Input tokens: 5,610 | Output tokens: 54]

Agent Response:
The status of your leave request MW19740 is pending. It is for 7 days of annual leave.


## 5. Full Workflow Tests

Test complete workflows from start to finish.


### Test 5.1: Complete Draft -> Submit Workflow


In [29]:
# Test 5.1: Complete workflow using tools directly
print("=== Complete Workflow Test ===\n")

# Step 1: Check balance
print("Step 1: Check balance")
balance = json.loads(lt.get_leave_balance(employee_id, "annual"))
print(f"Annual leave balance: {balance['balance']} days\n")

# Step 2: Create draft
print("Step 2: Create draft")
draft = json.loads(lt.draft_leave_request(
    employee_id, "annual", "2025-12-20", "2025-12-22"
))
workflow_draft_id = draft.get("draft_id")
print(f"Draft created: {workflow_draft_id}")
print(f"Days: {draft['draft']['days']}\n")

# Step 3: Submit draft
print("Step 3: Submit draft")
submit = json.loads(lt.submit_draft_leave_request(employee_id, workflow_draft_id))
workflow_request_id = submit.get("request_id")
print(f"Request submitted: {workflow_request_id}")
print(f"Status: {submit['leave_status']}\n")

# Step 4: Check status
print("Step 4: Check status")
status = json.loads(lt.check_leave_status(workflow_request_id))
print(f"Request {workflow_request_id} status: {status['status']}")
print(f"Days: {status['days']}, Type: {status['leave_type']}")


=== Complete Workflow Test ===

Step 1: Check balance
Annual leave balance: 14 days

Step 2: Create draft
--- SYSTEM: Draft created for mark_tan (annual leave) from 2025-12-20 to 2025-12-22 (3 days) ---
Draft created: draft-38795
Days: 3

Step 3: Submit draft
--- SYSTEM: Submitting draft draft-38795 as MW54365 - 3 days of annual leave for mark_tan (status: pending) ---
Request submitted: MW54365
Status: pending

Step 4: Check status
Request MW54365 status: pending
Days: 3, Type: annual


### Test 5.2: Agent Conversation Workflow


In [30]:
# Test 5.2: Simulate a conversation with the agent
print("=== Agent Conversation Workflow ===\n")

# Turn 1: User asks about balance
print("User: How many days of medical leave do I have?")
response1 = leave_smol_agent.run("How many days of medical leave do I have?")
print(f"Agent: {response1}\n")

# Turn 2: User requests leave with dates
print("User: I want to take 2 days of medical leave from 2025-11-18 to 2025-11-19")
response2 = leave_smol_agent.run("I want to take 2 days of medical leave from 2025-11-18 to 2025-11-19")
print(f"Agent: {response2}\n")

# Turn 3: User approves submission (if draft was created)
# Note: In a real scenario, the agent would wait for explicit approval
# For testing, we'll check if a draft was mentioned in the response
if "draft" in response2.lower() or "draft_id" in response2.lower():
    print("User: Yes, please submit it")
    # Extract draft_id from response if possible, or use a known one
    # In practice, the agent would maintain conversation context
    print("(Note: In a real conversation, the agent would extract the draft_id from context)")


=== Agent Conversation Workflow ===

User: How many days of medical leave do I have?


╭────────────────────────────────────────── New run - leave_specialist ───────────────────────────────────────────╮
│                                                                                                                 │
│ How many days of medical leave do I have?                                                                       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_leave_balance_tool' with arguments: {'employee_id': 'mark_tan', 'leave_type': 'medical'}     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"balance": 14, "unit": "days"}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': "I've created a draft leave request for 2 days of       │
│ medical leave from 2025-11-18 to 2025-11-19. Your draft ID is draft-51057. Would you like me to submit this     │
│ leave request for you?"}                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: I've created a draft leave request for 2 days of medical leave from 2025-11-18 to 2025-11-19. Your 
draft ID is draft-51057. Would you like me to submit this leave request for you?

Final answer: I've created a draft leave request for 2 days of medical leave from 2025-11-18 to 2025-11-19. Your 
draft ID is draft-51057. Would you like me to submit this leave request for you?

[Step 2: Duration 1.73 seconds| Input tokens: 5,724 | Output tokens: 109]

Agent: I've created a draft leave request for 2 days of medical leave from 2025-11-18 to 2025-11-19. Your draft ID is draft-51057. Would you like me to submit this leave request for you?

User: Yes, please submit it
(Note: In a real conversation, the agent would extract the draft_id from context)


## 6. Summary and Test Results

This section summarizes the test results and any issues found.


In [31]:
# Summary: Verify all core functions work
print("=== Test Summary ===\n")

tests_passed = []
tests_failed = []

# Test basic functionality
try:
    result = json.loads(lt.get_leave_balance("mark_tan", "annual"))
    if "balance" in result:
        tests_passed.append("get_leave_balance")
    else:
        tests_failed.append("get_leave_balance")
except Exception as e:
    tests_failed.append(f"get_leave_balance: {e}")

try:
    result = json.loads(lt.draft_leave_request("mark_tan", "annual", "2025-12-01", "2025-12-02"))
    if "draft_id" in result:
        tests_passed.append("draft_leave_request")
    else:
        tests_failed.append("draft_leave_request")
except Exception as e:
    tests_failed.append(f"draft_leave_request: {e}")

try:
    result = json.loads(lt.list_leave_drafts("mark_tan"))
    if "drafts" in result:
        tests_passed.append("list_leave_drafts")
    else:
        tests_failed.append("list_leave_drafts")
except Exception as e:
    tests_failed.append(f"list_leave_drafts: {e}")

try:
    result = json.loads(lt.submit_leave_request("mark_tan", "medical", 1))
    if "request_id" in result:
        tests_passed.append("submit_leave_request")
    else:
        tests_failed.append("submit_leave_request")
except Exception as e:
    tests_failed.append(f"submit_leave_request: {e}")

print(f"✓ Tests Passed: {len(tests_passed)}")
for test in tests_passed:
    print(f"  - {test}")

if tests_failed:
    print(f"\n✗ Tests Failed: {len(tests_failed)}")
    for test in tests_failed:
        print(f"  - {test}")
else:
    print("\n✓ All core functionality tests passed!")


=== Test Summary ===

--- SYSTEM: Draft created for mark_tan (annual leave) from 2025-12-01 to 2025-12-02 (2 days) ---
--- SYSTEM: Submitting 1 days of medical leave for mark_tan with request ID MW90491 (status: pending) ---
✓ Tests Passed: 4
  - get_leave_balance
  - draft_leave_request
  - list_leave_drafts
  - submit_leave_request

✓ All core functionality tests passed!
